# Data Exam (Hard) — CoderPad Practice Problems

Same format as INTERVIEW_EXAM.ipynb but harder. These problems have:
- More interacting components (3-4 functions/methods that depend on each other)
- Subtler edge cases
- More design decisions to make within the spec

Still 30-min timer, still all specs/formulas given, still realistic for a 45-min CoderPad.

**Do INTERVIEW_EXAM.ipynb first.** Come here when those feel comfortable.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import numpy as np
from typing import Any, Iterator
from collections import defaultdict
import math

---

## Problem 1: Near-Duplicate Detection Pipeline

### Scenario

Before training, you need to remove near-duplicate images from your dataset. You're given pre-computed embeddings for each image.

### Spec

**Cosine similarity:** `cos(a, b) = (a · b) / (||a|| * ||b||)`

Two items are **near-duplicates** if their cosine similarity >= `threshold`.

A **duplicate group** is a set of items where every item is a near-duplicate of at least one other item in the group (connected components — use union-find or BFS, your choice).

### Requirements

**`find_duplicate_pairs(embeddings: torch.Tensor, threshold: float) -> list[tuple[int, int]]`:**
- embeddings: (N, D)
- Return list of (i, j) pairs where i < j and cos_sim(i, j) >= threshold
- Use matrix operations, not nested loops

**`group_duplicates(num_items: int, pairs: list[tuple[int, int]]) -> list[set[int]]`:**
- Given duplicate pairs, find connected components
- Return only groups with 2+ items (singletons are unique, don't include them)

**`deduplicate(embeddings: torch.Tensor, quality_scores: torch.Tensor, threshold: float) -> dict`:**
- End-to-end: find pairs → group → for each group, keep the item with highest quality_score
- Return `{"kept": sorted list of kept indices, "removed": sorted list of removed indices, "groups": list of groups, "removal_rate": float}`

In [ ]:
def find_duplicate_pairs(embeddings: torch.Tensor, threshold: float) -> list[tuple[int, int]]:
    # YOUR CODE HERE
    pass


def group_duplicates(num_items: int, pairs: list[tuple[int, int]]) -> list[set[int]]:
    # YOUR CODE HERE
    pass


def deduplicate(embeddings: torch.Tensor, quality_scores: torch.Tensor, threshold: float) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 1 ---
torch.manual_seed(42)

# Create embeddings with known structure
base_a = F.normalize(torch.randn(1, 16), dim=1)
base_b = F.normalize(torch.randn(1, 16), dim=1)
unique = F.normalize(torch.randn(1, 16), dim=1)

embs = F.normalize(torch.cat([
    base_a + 0.01 * torch.randn(1, 16),  # 0 — group A
    base_a + 0.01 * torch.randn(1, 16),  # 1 — group A
    base_a + 0.01 * torch.randn(1, 16),  # 2 — group A
    base_b + 0.01 * torch.randn(1, 16),  # 3 — group B
    base_b + 0.01 * torch.randn(1, 16),  # 4 — group B
    unique,                                # 5 — unique
]), dim=1)
quality = torch.tensor([0.5, 0.9, 0.3, 0.7, 0.8, 0.6])

# find_duplicate_pairs
pairs = find_duplicate_pairs(embs, threshold=0.95)
assert all(i < j for i, j in pairs), "Pairs should have i < j"
# Items 0,1,2 should pair with each other; 3,4 should pair
pair_set = set(pairs)
assert (0, 1) in pair_set or (0, 2) in pair_set, "Group A pairs should be found"
assert (3, 4) in pair_set, "Group B pair should be found"
# Item 5 shouldn't pair with anyone
assert not any(5 in p for p in pairs), "Unique item shouldn't pair"

# group_duplicates
groups = group_duplicates(6, pairs)
assert len(groups) == 2, f"Expected 2 groups, got {len(groups)}"
all_grouped = set().union(*groups)
assert 5 not in all_grouped, "Unique item shouldn't be in any group"
group_a = [g for g in groups if 0 in g][0]
group_b = [g for g in groups if 3 in g][0]
assert group_a == {0, 1, 2}
assert group_b == {3, 4}

# deduplicate
result = deduplicate(embs, quality, threshold=0.95)
assert sorted(result["kept"]) == [1, 4, 5], f"Expected [1,4,5], got {sorted(result['kept'])}"
assert sorted(result["removed"]) == [0, 2, 3]
assert result["removal_rate"] == 3/6

# Edge: no duplicates
random_embs = F.normalize(torch.randn(5, 16), dim=1)
result_none = deduplicate(random_embs, torch.ones(5), threshold=0.99)
assert len(result_none["removed"]) == 0
assert len(result_none["kept"]) == 5

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Stratified Batch Sampler

### Scenario

Your dataset has class labels, and some classes are much rarer than others. You need a custom sampler that ensures every batch has balanced class representation.

### Requirements

**`StratifiedBatchSampler(Sampler)`:**
- `__init__(self, labels: list[int], batch_size: int, drop_last: bool = False)` 
- Each batch should have approximately equal numbers of each class
  - If batch_size=12 and 3 classes: 4 items from each class per batch
  - If batch_size doesn't divide evenly by num_classes, distribute remainder round-robin (class 0 gets one extra first, then class 1, etc.)
- Within each class, shuffle the order each time `__iter__` is called
- When a class runs out of items, resample from that class (oversample minority classes)
- If `drop_last=True`, don't yield the final batch if it would be smaller than batch_size
- `__iter__` yields lists of indices (one list per batch)
- `__len__` returns the number of batches
- Total items yielded per epoch = largest_class_size * num_classes (so every class contributes equally)

In [ ]:
class StratifiedBatchSampler(Sampler):
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 2 ---
torch.manual_seed(42)

# Imbalanced: class 0 has 20, class 1 has 5, class 2 has 10
labels = [0]*20 + [1]*5 + [2]*10

sampler = StratifiedBatchSampler(labels, batch_size=9, drop_last=False)

# Collect all batches
all_batches = list(sampler)
assert len(all_batches) > 0

# Each batch should have 3 items per class (9 / 3 classes)
for batch_idx, batch in enumerate(all_batches[:-1]):  # skip last which might be partial
    assert len(batch) == 9, f"Batch {batch_idx} has {len(batch)} items, expected 9"
    class_counts = defaultdict(int)
    for idx in batch:
        class_counts[labels[idx]] += 1
    assert class_counts[0] == 3, f"Batch {batch_idx}: class 0 has {class_counts[0]}, expected 3"
    assert class_counts[1] == 3, f"Batch {batch_idx}: class 1 has {class_counts[1]}, expected 3"
    assert class_counts[2] == 3, f"Batch {batch_idx}: class 2 has {class_counts[2]}, expected 3"

# Minority class (1, only 5 items) should be oversampled
all_class1_indices = []
for batch in all_batches:
    for idx in batch:
        if labels[idx] == 1:
            all_class1_indices.append(idx)
# Class 1 has only indices 20-24, but we need more than 5 appearances
# So some indices should repeat (oversampling)
assert len(all_class1_indices) > 5, "Class 1 should be oversampled"
assert all(20 <= idx <= 24 for idx in all_class1_indices), "Class 1 indices should be 20-24"

# Shuffling: two iterations should produce different orders
batches_1 = list(sampler)
batches_2 = list(sampler)
# Flatten
flat_1 = [idx for b in batches_1 for idx in b]
flat_2 = [idx for b in batches_2 for idx in b]
assert flat_1 != flat_2, "Should shuffle between iterations"

# drop_last
sampler_drop = StratifiedBatchSampler(labels, batch_size=9, drop_last=True)
batches_drop = list(sampler_drop)
assert all(len(b) == 9 for b in batches_drop), "drop_last should ensure all batches are full"

# Uneven batch_size (doesn't divide by num_classes)
sampler_uneven = StratifiedBatchSampler(labels, batch_size=10)
batches_uneven = list(sampler_uneven)
for batch in batches_uneven[:-1]:
    assert len(batch) == 10
    class_counts = defaultdict(int)
    for idx in batch:
        class_counts[labels[idx]] += 1
    # 10 / 3 classes = 3 each + 1 extra → should be [4, 3, 3] or similar
    counts = sorted(class_counts.values())
    assert counts[-1] - counts[0] <= 1, f"Class counts should differ by at most 1: {dict(class_counts)}"

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Caption Similarity and Deduplication

### Scenario

Your re-captioning pipeline generated new captions for images, but some captions are near-duplicates or suspiciously similar. Build a system to detect and report caption quality issues.

### Formulas (provided)

**Jaccard similarity** of two sets A, B: `|A ∩ B| / |A ∪ B|` (0 if both empty)

**Token overlap precision:** fraction of tokens in caption A that appear in caption B

**N-gram:** a contiguous subsequence of n tokens. E.g., for `["a", "red", "car"]`: bigrams are `[("a","red"), ("red","car")]`

### Requirements

**`tokenize_caption(text: str) -> list[str]`:**
- Lowercase, split on whitespace, remove any token that is purely punctuation

**`caption_similarity(cap_a: str, cap_b: str) -> dict`:**
- Return `{"jaccard": float, "precision_a_in_b": float, "precision_b_in_a": float, "bigram_overlap": float}`
- jaccard: Jaccard similarity of token sets
- precision_a_in_b: fraction of tokens in A that appear in B
- precision_b_in_a: fraction of tokens in B that appear in A
- bigram_overlap: Jaccard similarity of bigram sets
- Handle edge cases: empty captions should return all 0.0

**`find_similar_captions(captions: list[str], threshold: float) -> list[dict]`:**
- Find all pairs (i, j) where i < j and jaccard similarity >= threshold
- Return list of `{"i": int, "j": int, "similarity": dict}` sorted by jaccard descending

**`caption_diversity_report(captions: list[str]) -> dict`:**
- `"total_captions"`: int
- `"unique_captions"`: number of distinct captions (exact match after lowering)
- `"avg_length"`: mean token count
- `"vocabulary_size"`: total unique tokens across all captions
- `"most_common_tokens"`: list of (token, count) for top 5 most frequent tokens

In [ ]:
def tokenize_caption(text: str) -> list[str]:
    # YOUR CODE HERE
    pass


def caption_similarity(cap_a: str, cap_b: str) -> dict:
    # YOUR CODE HERE
    pass


def find_similar_captions(captions: list[str], threshold: float) -> list[dict]:
    # YOUR CODE HERE
    pass


def caption_diversity_report(captions: list[str]) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 3 ---

# tokenize_caption
assert tokenize_caption("A Red Car!") == ["a", "red", "car"]
assert tokenize_caption("hello, world.") == ["hello,", "world."]  # wait, "purely punctuation" only
# Actually: "hello," is not purely punctuation, so it stays. "," alone would be removed.
assert tokenize_caption("the . quick , fox") == ["the", "quick", "fox"]  # "." and "," are purely punct
assert tokenize_caption("") == []
assert tokenize_caption("  spaces  ") == ["spaces"]

# caption_similarity
sim = caption_similarity("a red car on the road", "a blue car on the highway")
# Tokens A: {a, red, car, on, the, road}, B: {a, blue, car, on, the, highway}
# Intersection: {a, car, on, the} = 4
# Union: {a, red, blue, car, on, the, road, highway} = 8
# Jaccard: 4/8 = 0.5
assert abs(sim["jaccard"] - 0.5) < 1e-5, f"Expected jaccard=0.5, got {sim['jaccard']}"
assert abs(sim["precision_a_in_b"] - 4/6) < 1e-5  # 4 of 6 A-tokens appear in B
assert abs(sim["precision_b_in_a"] - 4/6) < 1e-5

# Bigram overlap
# A bigrams: (a,red),(red,car),(car,on),(on,the),(the,road)
# B bigrams: (a,blue),(blue,car),(car,on),(on,the),(the,highway)
# Intersection: {(car,on),(on,the)} = 2
# Union: 8
assert abs(sim["bigram_overlap"] - 2/8) < 1e-5

# Identical captions
sim_id = caption_similarity("same words here", "same words here")
assert abs(sim_id["jaccard"] - 1.0) < 1e-5

# Empty
sim_empty = caption_similarity("", "hello")
assert sim_empty["jaccard"] == 0.0
sim_both_empty = caption_similarity("", "")
assert sim_both_empty["jaccard"] == 0.0

# find_similar_captions
captions = [
    "a red car on the road",        # 0
    "a red car on the highway",      # 1 — similar to 0
    "a dog playing in the park",     # 2 — different
    "a red car on the road today",   # 3 — very similar to 0
    "a cat sleeping on the couch",   # 4 — different
]
similar = find_similar_captions(captions, threshold=0.5)
assert len(similar) > 0
# (0,1), (0,3), and (1,3) should all be found
found_pairs = {(s["i"], s["j"]) for s in similar}
assert (0, 3) in found_pairs or (0, 1) in found_pairs, "Should find similar car captions"
# Should be sorted by jaccard descending
jaccards = [s["similarity"]["jaccard"] for s in similar]
assert jaccards == sorted(jaccards, reverse=True), "Should be sorted by jaccard desc"

# caption_diversity_report
report = caption_diversity_report(captions)
assert report["total_captions"] == 5
assert report["unique_captions"] == 5  # all different
assert report["avg_length"] > 0
assert report["vocabulary_size"] > 0
assert len(report["most_common_tokens"]) == 5
# "the" and "a" should be among most common
top_tokens = {t[0] for t in report["most_common_tokens"]}
assert "a" in top_tokens or "the" in top_tokens

# Exact duplicate detection
captions_with_dup = ["hello world", "foo bar", "hello world", "baz"]
report_dup = caption_diversity_report(captions_with_dup)
assert report_dup["unique_captions"] == 3  # "hello world" appears twice

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Bucketed DataLoader for Variable-Length Sequences

### Scenario

Video clips and captions have varying lengths. Naive batching wastes compute on padding. Build a bucketed loader that groups similar-length items together.

### Requirements

**`BucketedDataset(Dataset)`:**
- `__init__(self, records: list[dict], num_buckets: int)` — records have `"id"` (str), `"tokens"` (list[int]), `"label"` (int)
- Sort records by token length, divide into `num_buckets` equal-sized buckets
- `__getitem__` returns `{"id": str, "tokens": int64 tensor, "label": int64 scalar tensor}`
- `__len__` returns total records
- `get_bucket_boundaries(self) -> list[tuple[int, int]]` — return list of (min_length, max_length) for each bucket

**`bucket_collate(batch: list[dict]) -> dict`:**
- Same as standard padded collation, but the padding waste should be lower since items in a batch have similar lengths
- Returns `{"ids": list[str], "tokens": (B, max_len) int64, "mask": (B, max_len) int64, "labels": (B,) int64}`
- Pad with 0

**`compute_padding_efficiency(batches: list[dict]) -> dict`:**
- Given a list of collated batches, compute:
  - `"total_tokens"`: sum of all real (non-padding) tokens across all batches
  - `"total_cells"`: sum of batch_size * max_len across all batches (total tensor cells)
  - `"efficiency"`: total_tokens / total_cells (1.0 = no wasted padding)

In [ ]:
class BucketedDataset(Dataset):
    # YOUR CODE HERE
    pass


def bucket_collate(batch: list[dict]) -> dict:
    # YOUR CODE HERE
    pass


def compute_padding_efficiency(batches: list[dict]) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 4 ---
torch.manual_seed(42)

# Create records with varying lengths
records = []
for i in range(100):
    length = np.random.randint(5, 50)  # lengths from 5 to 49
    records.append({
        "id": f"item_{i}",
        "tokens": list(np.random.randint(1, 1000, length)),
        "label": np.random.randint(0, 3),
    })

ds = BucketedDataset(records, num_buckets=5)
assert len(ds) == 100

# Check bucket boundaries
boundaries = ds.get_bucket_boundaries()
assert len(boundaries) == 5
# Boundaries should be sorted (buckets go from short to long)
for i in range(len(boundaries) - 1):
    assert boundaries[i][1] <= boundaries[i+1][0] or boundaries[i][1] <= boundaries[i+1][1], \
        f"Buckets should be ordered by length: {boundaries}"

# Single item
item = ds[0]
assert isinstance(item["id"], str)
assert item["tokens"].dtype == torch.int64
assert item["label"].dtype == torch.int64
assert item["label"].ndim == 0  # scalar

# Collation
batch = [ds[i] for i in range(4)]
collated = bucket_collate(batch)
assert collated["tokens"].shape[0] == 4
assert collated["mask"].shape == collated["tokens"].shape
assert collated["labels"].shape == (4,)
assert len(collated["ids"]) == 4

# Mask correctness
for i, item in enumerate(batch):
    orig_len = len(item["tokens"])
    assert collated["mask"][i].sum().item() == orig_len

# Padding efficiency: bucketed should be better than random
# Random batches
random_loader = DataLoader(ds, batch_size=10, collate_fn=bucket_collate, shuffle=True)
random_batches = [b for b in random_loader]

# Sequential (bucketed) batches — items near each other have similar lengths
bucketed_loader = DataLoader(ds, batch_size=10, collate_fn=bucket_collate, shuffle=False)
bucketed_batches = [b for b in bucketed_loader]

random_eff = compute_padding_efficiency(random_batches)
bucketed_eff = compute_padding_efficiency(bucketed_batches)

assert 0 < random_eff["efficiency"] <= 1.0
assert 0 < bucketed_eff["efficiency"] <= 1.0
assert bucketed_eff["efficiency"] >= random_eff["efficiency"] - 0.1, \
    f"Bucketed ({bucketed_eff['efficiency']:.3f}) should be at least as efficient as random ({random_eff['efficiency']:.3f})"

print(f"Random efficiency: {random_eff['efficiency']:.3f}")
print(f"Bucketed efficiency: {bucketed_eff['efficiency']:.3f}")
print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Multi-Stage Quality Filter

### Scenario

Runway uses hierarchical quality filtering — loose filters for early training, strict for later stages. Build a configurable multi-stage filter.

### Requirements

**`QualityFilter`:**
- `__init__(self, rules: list[dict])` — each rule is a dict with:
  - `"field"`: str — key in the record dict
  - `"op"`: str — one of `"gte"`, `"lte"`, `"gt"`, `"lt"`, `"eq"`, `"in"`, `"not_in"`, `"contains"`
  - `"value"`: the threshold/set to compare against
  - `"required"`: bool (default True) — if the field is missing, does the record fail?
- `apply(self, record: dict) -> bool` — True if record passes ALL rules
- `apply_batch(self, records: list[dict]) -> tuple[list[dict], list[dict]]` — return (passed, failed)
- `report(self, records: list[dict]) -> dict` — return `{"total": int, "passed": int, "failed": int, "failure_reasons": dict}` where failure_reasons maps rule description to count of records that failed that specific rule

**`FilterPipeline`:**
- `__init__(self, stages: list[tuple[str, QualityFilter]])` — named stages applied in order
- `run(self, records: list[dict]) -> dict` — apply each stage sequentially (output of stage N is input to stage N+1)
  - Return `{"final_records": list[dict], "stage_reports": list[dict]}` where each stage report includes the stage name and the QualityFilter report for that stage

In [ ]:
class QualityFilter:
    # YOUR CODE HERE
    pass


class FilterPipeline:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 5 ---

# Basic QualityFilter
qf = QualityFilter([
    {"field": "quality_score", "op": "gte", "value": 0.5},
    {"field": "width", "op": "gte", "value": 256},
    {"field": "format", "op": "in", "value": {"jpg", "png", "webp"}},
])

records = [
    {"id": "a", "quality_score": 0.9, "width": 512, "format": "jpg"},    # pass
    {"id": "b", "quality_score": 0.3, "width": 1024, "format": "png"},   # fail: quality
    {"id": "c", "quality_score": 0.7, "width": 128, "format": "jpg"},    # fail: width
    {"id": "d", "quality_score": 0.8, "width": 512, "format": "bmp"},    # fail: format
    {"id": "e", "quality_score": 0.6, "width": 256, "format": "webp"},   # pass
]

passed, failed = qf.apply_batch(records)
passed_ids = {r["id"] for r in passed}
assert passed_ids == {"a", "e"}, f"Expected {{a, e}}, got {passed_ids}"

# Report
report = qf.report(records)
assert report["total"] == 5
assert report["passed"] == 2
assert report["failed"] == 3
assert len(report["failure_reasons"]) > 0

# Missing field — required=True by default
qf_req = QualityFilter([{"field": "score", "op": "gte", "value": 0.5}])
assert qf_req.apply({"score": 0.8}) == True
assert qf_req.apply({"other": 0.8}) == False  # missing required field

# Missing field — required=False
qf_opt = QualityFilter([{"field": "score", "op": "gte", "value": 0.5, "required": False}])
assert qf_opt.apply({"other": 0.8}) == True  # missing optional field passes

# "contains" operator
qf_contains = QualityFilter([{"field": "caption", "op": "contains", "value": "car"}])
assert qf_contains.apply({"caption": "a red car"}) == True
assert qf_contains.apply({"caption": "a dog"}) == False

# "not_in" operator
qf_notin = QualityFilter([{"field": "source", "op": "not_in", "value": {"spam", "nsfw"}}])
assert qf_notin.apply({"source": "web"}) == True
assert qf_notin.apply({"source": "spam"}) == False

# FilterPipeline
pipeline = FilterPipeline([
    ("coarse", QualityFilter([
        {"field": "quality_score", "op": "gte", "value": 0.3},
    ])),
    ("fine", QualityFilter([
        {"field": "quality_score", "op": "gte", "value": 0.7},
        {"field": "width", "op": "gte", "value": 256},
    ])),
])

all_records = [
    {"id": "1", "quality_score": 0.9, "width": 512},   # survives both
    {"id": "2", "quality_score": 0.1, "width": 1024},  # fails coarse
    {"id": "3", "quality_score": 0.5, "width": 512},   # passes coarse, fails fine (quality)
    {"id": "4", "quality_score": 0.8, "width": 128},   # passes coarse, fails fine (width)
    {"id": "5", "quality_score": 0.75, "width": 256},  # survives both
]

result = pipeline.run(all_records)
final_ids = {r["id"] for r in result["final_records"]}
assert final_ids == {"1", "5"}, f"Expected {{1, 5}}, got {final_ids}"
assert len(result["stage_reports"]) == 2
assert result["stage_reports"][0]["total"] == 5  # coarse sees all 5
assert result["stage_reports"][1]["total"] == 4  # fine sees 4 (after coarse removed 1)

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: Feature Extractor with Caching

### Scenario

Computing embeddings is expensive. Build a feature extraction wrapper that caches results and supports batch processing.

### Requirements

**`CachedFeatureExtractor`:**
- `__init__(self, model_fn, cache_size: int)` — `model_fn` takes (B, input_dim) tensor, returns (B, output_dim) tensor. `cache_size` is max items to cache (LRU eviction).
- `extract(self, ids: list[str], inputs: torch.Tensor) -> torch.Tensor`:
  - inputs: (N, input_dim)
  - For each item, check if its ID is in cache. If so, use cached result.
  - For uncached items, batch them together and call model_fn once.
  - Cache the new results.
  - Return (N, output_dim) in the same order as the input IDs.
- `cache_hit_rate` property — fraction of total extract() calls that were cache hits
- `clear_cache(self)` — empty the cache and reset stats

**Important:**
- If all items are cached, don't call model_fn at all
- If no items are cached, call model_fn once with all inputs
- Maintain insertion order for LRU: when cache is full, evict the least recently accessed item
- Accessing a cached item counts as a "use" and should update its recency

In [ ]:
class CachedFeatureExtractor:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 6 ---

# Track how many times model_fn is called
call_log = []
def mock_model(x: torch.Tensor) -> torch.Tensor:
    call_log.append(x.shape[0])
    return x @ torch.randn(8, 4)  # (B, 8) -> (B, 4)

extractor = CachedFeatureExtractor(mock_model, cache_size=5)

# First call — all miss
call_log.clear()
inputs = torch.randn(3, 8)
out1 = extractor.extract(["a", "b", "c"], inputs)
assert out1.shape == (3, 4)
assert len(call_log) == 1, "Should call model once for all misses"
assert call_log[0] == 3, "Should process 3 items in one batch"

# Second call — all hits
call_log.clear()
out2 = extractor.extract(["a", "b", "c"], inputs)  # same IDs
assert len(call_log) == 0, "Should not call model when all cached"
assert torch.allclose(out1, out2), "Cached results should match"

# Partial hits
call_log.clear()
inputs_mixed = torch.randn(4, 8)
out3 = extractor.extract(["a", "d", "b", "e"], inputs_mixed)
assert out3.shape == (4, 4)
assert len(call_log) == 1, "Should call model once for misses"
assert call_log[0] == 2, "Should only process 2 uncached items"
# a and b should come from cache (same values as before)
assert torch.allclose(out3[0], out1[0]), "'a' should be cached"
assert torch.allclose(out3[2], out1[1]), "'b' should be cached"

# Cache hit rate
# Calls so far: [a,b,c] (0/3 hits) + [a,b,c] (3/3 hits) + [a,d,b,e] (2/4 hits)
# Total: 10 lookups, 5 hits
assert abs(extractor.cache_hit_rate - 5/10) < 1e-5, f"Expected 0.5, got {extractor.cache_hit_rate}"

# LRU eviction — cache_size=5, currently has [a,b,c,d,e]
call_log.clear()
new_inputs = torch.randn(2, 8)
extractor.extract(["f", "g"], new_inputs)  # should evict oldest (c was least recently used)
assert len(call_log) == 1

# Verify eviction: 'c' should be evicted (oldest not recently accessed)
# a, b were accessed in the partial hits call, so they're more recent
call_log.clear()
extractor.extract(["a"], torch.randn(1, 8))  # should be cached still
assert len(call_log) == 0, "'a' should still be in cache (recently accessed)"

# Clear
extractor.clear_cache()
call_log.clear()
extractor.extract(["a"], torch.randn(1, 8))
assert len(call_log) == 1, "After clear, should miss"

print("Problem 6: ALL TESTS PASSED")

---

## Problem 7: Adaptive Layer Norm + Residual Block

### Scenario

DiT (Diffusion Transformer) conditions on timestep by modulating LayerNorm parameters. Build the building block.

### Formulas (provided)

**Standard LayerNorm:** `LN(x) = gamma * (x - mean) / sqrt(var + eps) + beta`

**Adaptive LayerNorm (adaLN):** Instead of learned gamma/beta, predict them from a conditioning signal:
```
gamma, beta = MLP(condition).chunk(2)   # condition -> 2*dim
adaLN(x) = (1 + gamma) * LN(x) + beta  # note the (1 + gamma) — it's a residual scale
```

### Requirements

**`AdaptiveLayerNorm(nn.Module)`:**
- `__init__(self, dim: int, cond_dim: int)` — dim is feature dim, cond_dim is conditioning dim
- Internally uses a standard `nn.LayerNorm(dim)` (no learned affine — set `elementwise_affine=False`)
- Has a linear layer: cond_dim → 2 * dim (for gamma and beta)
- `forward(self, x, cond)` — x: (B, T, dim), cond: (B, cond_dim) → output: (B, T, dim)
  - Broadcast cond across the T dimension

**`ResidualBlock(nn.Module)`:**
- `__init__(self, dim: int, cond_dim: int, mlp_ratio: int = 4)`
- Architecture: `x + MLP(adaLN(x, cond))`
- MLP: Linear(dim, dim*mlp_ratio) → GELU → Linear(dim*mlp_ratio, dim)
- `forward(self, x, cond)` — same shapes as AdaptiveLayerNorm

In [ ]:
class AdaptiveLayerNorm(nn.Module):
    # YOUR CODE HERE
    pass


class ResidualBlock(nn.Module):
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 7 ---
torch.manual_seed(42)

# AdaptiveLayerNorm
aln = AdaptiveLayerNorm(dim=64, cond_dim=32)
x = torch.randn(2, 10, 64)
cond = torch.randn(2, 32)
out = aln(x, cond)
assert out.shape == (2, 10, 64), f"Expected (2,10,64), got {out.shape}"

# Different conditions should produce different outputs
cond2 = torch.randn(2, 32)
out2 = aln(x, cond2)
assert not torch.allclose(out, out2, atol=1e-3), "Different conditions should give different outputs"

# Same condition should produce same output
out3 = aln(x, cond)
assert torch.allclose(out, out3), "Same inputs should give same output"

# ResidualBlock
rb = ResidualBlock(dim=64, cond_dim=32, mlp_ratio=4)
out_rb = rb(x, cond)
assert out_rb.shape == (2, 10, 64)

# Residual connection: output should be close to input (at init, MLP outputs are small)
# This is a soft check — at random init, the residual shouldn't dominate
diff = (out_rb - x).abs().mean()
input_scale = x.abs().mean()
assert diff < input_scale * 5, "Residual block should not explode at init"

# Gradient flows through both paths
loss = out_rb.sum()
loss.backward()
for name, p in rb.named_parameters():
    assert p.grad is not None, f"No gradient for {name}"
    assert not torch.isnan(p.grad).any(), f"NaN gradient for {name}"

# Conditioning affects output
out_rb2 = rb(x, cond2)
assert not torch.allclose(out_rb, out_rb2, atol=1e-3)

# Parameter count check
# adaLN: cond_dim * 2*dim = 32*128
# MLP: dim*dim*mlp_ratio + dim*mlp_ratio (bias) + dim*mlp_ratio*dim + dim (bias)
#     = 64*256 + 256 + 256*64 + 64 = 16384 + 256 + 16384 + 64 = 33088
# Total MLP: 33088, adaLN linear: 32*128 + 128 = 4224
total_params = sum(p.numel() for p in rb.parameters())
assert total_params > 0

print("Problem 7: ALL TESTS PASSED")

---

## Problem 8: Dataset Pipeline Orchestrator

### Scenario

You need to process a dataset through a series of stages (download, filter, embed, deduplicate, pack). Build a pipeline runner that handles stage dependencies, caching, and error reporting.

### Requirements

**`PipelineStage`:**
- `__init__(self, name: str, fn: callable, depends_on: list[str] | None = None)`
- `fn` takes a dict (accumulated results from previous stages) and returns a result value

**`Pipeline`:**
- `__init__(self)` — empty pipeline
- `add_stage(self, stage: PipelineStage)` — add a stage. Raise ValueError if a duplicate name or if depends_on references an unknown stage.
- `run(self) -> dict`:
  - Execute stages in dependency order (topological sort)
  - Each stage's fn receives a dict of all completed stage results
  - If a stage raises an exception, record the error and skip any stages that depend on it
  - Return `{"results": dict (stage_name → result), "errors": dict (stage_name → error message), "skipped": list[str], "execution_order": list[str]}`
- `validate(self) -> list[str]` — return list of issues (circular dependencies, missing dependencies). Empty list = valid.

In [ ]:
class PipelineStage:
    def __init__(self, name: str, fn: callable, depends_on: list[str] | None = None):
        self.name = name
        self.fn = fn
        self.depends_on = depends_on or []


class Pipeline:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 8 ---

# Basic pipeline
p = Pipeline()
p.add_stage(PipelineStage("load", lambda ctx: list(range(100))))
p.add_stage(PipelineStage("filter", lambda ctx: [x for x in ctx["load"] if x % 2 == 0], depends_on=["load"]))
p.add_stage(PipelineStage("count", lambda ctx: len(ctx["filter"]), depends_on=["filter"]))

result = p.run()
assert result["results"]["load"] == list(range(100))
assert result["results"]["filter"] == list(range(0, 100, 2))
assert result["results"]["count"] == 50
assert len(result["errors"]) == 0
assert len(result["skipped"]) == 0

# Execution order respects dependencies
order = result["execution_order"]
assert order.index("load") < order.index("filter")
assert order.index("filter") < order.index("count")

# Error handling: stage fails → dependents are skipped
p2 = Pipeline()
p2.add_stage(PipelineStage("load", lambda ctx: list(range(10))))
p2.add_stage(PipelineStage("bad_stage", lambda ctx: 1/0, depends_on=["load"]))  # will fail
p2.add_stage(PipelineStage("downstream", lambda ctx: len(ctx["bad_stage"]), depends_on=["bad_stage"]))
p2.add_stage(PipelineStage("independent", lambda ctx: "I'm fine"))  # no dependency on bad_stage

result2 = p2.run()
assert "load" in result2["results"]  # succeeded
assert "bad_stage" in result2["errors"]  # failed
assert "downstream" in result2["skipped"]  # skipped because depends on failed stage
assert result2["results"]["independent"] == "I'm fine"  # independent stage still runs

# Duplicate name
p3 = Pipeline()
p3.add_stage(PipelineStage("a", lambda ctx: 1))
try:
    p3.add_stage(PipelineStage("a", lambda ctx: 2))
    assert False, "Should reject duplicate name"
except ValueError:
    pass

# Unknown dependency
p4 = Pipeline()
try:
    p4.add_stage(PipelineStage("b", lambda ctx: 1, depends_on=["nonexistent"]))
    assert False, "Should reject unknown dependency"
except ValueError:
    pass

# Diamond dependency: A → B, A → C, B+C → D
p5 = Pipeline()
p5.add_stage(PipelineStage("A", lambda ctx: 10))
p5.add_stage(PipelineStage("B", lambda ctx: ctx["A"] * 2, depends_on=["A"]))
p5.add_stage(PipelineStage("C", lambda ctx: ctx["A"] + 5, depends_on=["A"]))
p5.add_stage(PipelineStage("D", lambda ctx: ctx["B"] + ctx["C"], depends_on=["B", "C"]))

result5 = p5.run()
assert result5["results"]["D"] == 35  # (10*2) + (10+5)
assert result5["execution_order"].index("A") < result5["execution_order"].index("B")
assert result5["execution_order"].index("A") < result5["execution_order"].index("C")
assert result5["execution_order"].index("B") < result5["execution_order"].index("D")
assert result5["execution_order"].index("C") < result5["execution_order"].index("D")

print("Problem 8: ALL TESTS PASSED")

---

## Scoring

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Strong pass** |
| Solved in 25-30 min, minor debugging | **Pass** — exam-ready |
| 30-40 min or needed hints | **Borderline** — redo tomorrow |
| Couldn't solve or > 40 min | **Go back to INTERVIEW_EXAM.ipynb** — nail the fundamentals first |